# 02- Preprocesado de datos y decisiones metodlógicas

**Proyecto Final Machine Learning**  
Adriana Lorenzo Mingorance

En este notebook se preparan los datos que se utilizarán en las tareas de clasificación y regresión.
Justificamos las variables excluidas, transformamos las variabels categóricas y realizamos la división train/test



## 0. Imports

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

## 1. Carga del dataset

In [4]:
df = pd.read_csv('../data/rendimiento_estudiantes.csv', sep=';')
print(f"Forma dataset: {df.shape}")
print(f"Nulos totales: {df.isnull().sum().sum()}")

df.head()

Forma dataset: (4424, 37)
Nulos totales: 0


,estado_civil,modo_solicitud,orden_solicitud,curso,asistencia_diurna_vespertina,cualificacion_previa,nota_cualificacion_previa,nacionalidad,cualificacion_madre,cualificacion_padre,...,asignaturas_2sem_convalidadas,asignaturas_2sem_matriculadas,asignaturas_2sem_evaluadas,asignaturas_2sem_aprobadas,nota_media_2sem,asignaturas_2sem_sin_evaluacion,tasa_desempleo,tasa_inflacion,pib,objetivo
0,soltero/a,2a_fase_contingente_general,5,animacion_y_diseno_multimedia,diurna,educacion_secundaria,122.0,portuguesa,educacion_basica_3er_ciclo_o_equivalente,otro_11o_ano,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,abandono
1,soltero/a,estudiante_internacional_licenciatura,1,turismo,diurna,educacion_secundaria,160.0,portuguesa,educacion_secundaria_12o_ano_o_equivalente,educacion_superior_grado,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,graduado
2,soltero/a,1a_fase_contingente_general,5,diseno_de_comunicacion,diurna,educacion_secundaria,122.0,portuguesa,educacion_basica_1er_ciclo_o_equivalente,educacion_basica_1er_ciclo_o_equivalente,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,abandono
3,soltero/a,2a_fase_contingente_general,2,periodismo_y_comunicacion,diurna,educacion_secundaria,122.0,portuguesa,educacion_basica_2o_ciclo_o_equivalente,educacion_basica_1er_ciclo_o_equivalente,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,graduado
4,casado/a,mayor_de_23_anos,1,servicio_social_nocturno,vespertina,educacion_secundaria,100.0,portuguesa,educacion_basica_1er_ciclo_o_equivalente,educacion_basica_2o_ciclo_o_equivalente,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,graduado


## 2. Definición target. Separamos los dos targets

In [5]:
#Para clasificación: situación final del estudiante
y_clf = df['objetivo']

#Para regresión: nota media 2º semestre
y_reg = df['nota_media_2sem']

print(f"Clases clasificación: {y_clf.value_counts().to_dict()}")
print(f"Target regresión-media: {y_reg.mean():2f}, std: {y_reg.std():.2f}")

Clases clasificación: {'graduado': 2209, 'abandono': 1421, 'matriculado': 794}
Target regresión-media: 10.230206, std: 5.21


## 3. Variables excluidas

Para clasificación excluimos solo la variable objetivo. Para regresión excluímos, además del target, las variables del segundo semestre que contienen información relevante para la nota que se quiere predecir

In [6]:
excluir_clf = ['objetivo']

#para regresión: excluimos variabels del 2º semestre con riesgo de leakage
excluir_reg = [
    'objetivo', 'nota_media_2sem', 'asignaturas_2sem_matriculadas', 'asignaturas_2sem_evaluadas', 'asignaturas_2sem_aprobadas', 'asignaturas_2sem_sin_evaluacion', 'asignaturas_2sem_convalidadas']

print("Variables excluidas por data leakage en regresión: ")
for v in excluir_reg:
    print(f" - {v}")

Variables excluidas por data leakage en regresión: 
 - objetivo
 - nota_media_2sem
 - asignaturas_2sem_matriculadas
 - asignaturas_2sem_evaluadas
 - asignaturas_2sem_aprobadas
 - asignaturas_2sem_sin_evaluacion
 - asignaturas_2sem_convalidadas


## 4. Codificación de variables categóricas

In [8]:
df_proc = df.copy() #creamos una copia para facilitarlo

vars_cat = df_proc.select_dtypes(include='object').columns.tolist()
vars_cat = [c for c in vars_cat if c!= 'objetivo']
print(f"Variables categóricas a codificar: {vars_cat}")

df_proc = pd.get_dummies(df_proc, columns=vars_cat, drop_first=True)
print(f"\nShape tras la codificación: {df_proc.shape}")

Variables categóricas a codificar: ['estado_civil', 'modo_solicitud', 'curso', 'asistencia_diurna_vespertina', 'cualificacion_previa', 'nacionalidad', 'cualificacion_madre', 'cualificacion_padre', 'ocupacion_madre', 'ocupacion_padre', 'desplazado', 'necesidades_educativas_especiales', 'deudor', 'matricula_al_dia', 'genero', 'becado', 'internacional']

Shape tras la codificación: (4424, 239)


Las variables categóricas se han transformado mediante one-hot encoding. Hemos aplicado 'drop_first=True', para evitar redundancia entre categóricas y facilitar el uso posterior en modelos lineales.

## 5. Construcción de datasets para cada tarea

In [11]:
#clasificacion
X_clf = df_proc.drop(columns=excluir_clf + ['nota_media_2sem']) #excluimos tambien nota 2 sem porque puede contaminar la prediccion de la situacion final
#regresión
X_reg = df_proc.drop(columns=excluir_reg) 

print("Clasificación: ")
print(f"Variables: {X_clf.shape[1]}")
print("\nRegresión: ")
print(f"Variables: {X_reg.shape[1]}")

Clasificación: 
Variables: 237

Regresión: 
Variables: 232


## 6. División train/test

In [12]:
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf)

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

print("CLasificación: ")
print(f"Train: {X_train_clf.shape}. | Test: {X_test_clf.shape} ")
print(f"Distribución train: {y_train_clf.value_counts(normalize=True).round(3).to_dict()}")
print(f"Distribución test: {y_test_clf.value_counts(normalize=True).round(3).to_dict()}")

print("\nRegresión: ")
print(f"Train: {X_train_reg.shape} | Test: {X_test_reg.shape}")

CLasificación: 
Train: (3539, 237). | Test: (885, 237) 
Distribución train: {'graduado': 0.499, 'abandono': 0.321, 'matriculado': 0.179}
Distribución test: {'graduado': 0.499, 'abandono': 0.321, 'matriculado': 0.18}

Regresión: 
Train: (3539, 232) | Test: (885, 232)


## 7. Escalado (para modelos que lo neceisten)

In [ ]:
from sklearn.preprocessing import StandardScaler #como hemos aprendido en las practicas de LDA
import pickle #añadimos apra guardar objetos de python en un archivo

#clasificacion
scaler_clf = StandardScaler()

X_train_clf_sc = pd.DataFrame(scaler_clf.fit_transform(X_train_clf), columns=X_train_clf.columns, index= X_train_clf.index)

X_test_cls_sc = pd.DataFrame(scaler_clf.transform(X_test_clf), columns=X_test_clf.columns, index= X_test_clf.index)


#regresion
scaler_reg = StandardScaler()

X_train_reg_sc = pd.DataFrame(scaler_clf.fit_transform(X_train_reg), columns=X_train_reg.columns, index= X_train_reg.index)

X_test_reg_sc = pd.DataFrame(scaler_clf.transform(X_test_reg), columns=X_test_reg.columns, index= X_test_reg.index)

#guardamos scalers con pickle
with open('../data/scaler_clf.pkl', 'wb') as f:
    pickle.dump(scaler_clf, f)

with open('../data/scaler_reg.pkl', 'wb') as f:
    pickle.dump(scaler_reg, f)

print("Scalers guardados")


Scalers guardados
